# 🦺 Yapay Zekâ Destekli Baret Tanıma Sistemi — Model Eğitimi (v2)

Bu notebook baretsiz kişi tespitini iyileştirmek için yeniden düzenlendi:

| Sorun | Bu sürümdeki çözüm |
|---|---|
| İkinci veri setinin etiketleri ters (Helmet=0, No Helmet=1) birleştiriliyordu | Etiketler **sınıf adına göre** yeniden eşleniyor (remap). Ters etiket artık modele giremez |
| Aynı dosya adları birbirinin üzerine yazılıyordu | Her dosyaya kaynak ön eki ekleniyor, kopyalar (MD5) ayıklanıyor |
| 5 sınıf tanımlı, 3'ünün örneği yoktu | Sadece **2 sınıf**: `head` (baretsiz) ve `helmet` (baretli). Hi-viz baret → `helmet` |
| Baret örnekleri azdı (2.032'ye karşı 684) | Eğitim listesinde baretli görseller dengeleniyor (oversampling) |
| Başörtülü / yarım yüz örnekleri yoktu | **Ekip veri seti** yuvası (kendi çektiğiniz videolar) |
| Overfitting kontrolü yoktu | Early stopping, augmentation, train/val loss grafiği |
| Eşik tahminle seçiliyordu | Test setinde Precision–Recall eğrisinden **önerilen eşik** hesaplanıyor |

> ⚙️ Çalıştırmadan önce: **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU**

In [ ]:
!pip install -q ultralytics roboflow

## 1. Roboflow API anahtarı (güvenli)
Anahtar **koda yazılmaz**. Colab'da sol menüdeki 🔑 **Secrets** bölümüne `ROBOFLOW_API_KEY` adıyla ekleyin ve "Notebook access" iznini açın.
> Eski notebook'taki anahtar public repoda açık kaldığı için Roboflow hesabından **iptal edilip yenilenmelidir**.

In [ ]:
from google.colab import userdata
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
assert ROBOFLOW_API_KEY, "Secrets bölümüne ROBOFLOW_API_KEY ekleyin."

## 2. Veri setleri

- `KAYNAKLAR`: Roboflow Universe veri setleri. Çeşitliliği artırmak için listeye yeni veri seti eklenebilir. Sınıf adları otomatik eşlendiği için etiket sırası önemli değildir.
- `EKIP_VERI_SETI`: Ekibin kendi çektiği videolardan oluşturduğu Roboflow projesi. **%95 hedefi için en kritik kaynak budur.**

**Ekip veri seti etiketleme kuralları** (sadece 2 sınıf: `head`, `helmet`):
1. Baretli kişi → `helmet` (başörtülü + baretli dahil)
2. Baretsiz kişi → `head` (**başörtülü + baretsiz dahil**, İSG açısından ihlaldir)
3. Şapka, bere, kapüşon, başörtüsü takan **baretsiz** kişiler bol miktarda eklenmeli. Model "saç yok = baret" ezberini bu örneklerle bırakır.
4. Kadraj kenarından yarısından fazlası kesilmiş kafa veya sadece alt yüz görünen kareler **etiketlenmez**. Model bu durumlarda karar üretmemeyi öğrenir; uygulama da bu durumda beklemede kalır.
5. Videolar farklı ışık (gün ışığı, floresan, loş), mesafe (1–5 m), açı ve 1–4 kişilik sahneleri kapsamalı.
6. Roboflow'da **Test** bölümüne eğitimde hiç kullanılmayacak *ayrı videoların* kareleri konulmalı. Aynı videonun kareleri hem train hem test'e düşerse sonuç yapay olarak şişer.

In [ ]:
# (workspace, project, version) — yeni kaynak eklemek için listeye satır ekleyin
KAYNAKLAR = [
    ("universe-datasets",    "hard-hat-universe-0dy7t", 26),
    ("construction-helmets", "safety_helmets",          1),   # etiketleri ters, aşağıda isimle düzeltiliyor
]

# Ekip veri seti hazır olunca doldurun, örn: ("ekip-workspace", "baret-saha-verisi", 1)
EKIP_VERI_SETI = None

MERGED_DIR = "/content/birlesik_veri"

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
indirilenler = []   # (kaynak_adı, klasör, ekip_mi)

for ws, proj, ver in KAYNAKLAR:
    try:
        ds = rf.workspace(ws).project(proj).version(ver).download("yolov8")
        indirilenler.append((proj, ds.location, False))
        print(f"✅ {proj} v{ver} indirildi")
    except Exception as e:
        print(f"⚠️ {proj} indirilemedi: {e}")

if EKIP_VERI_SETI:
    ws, proj, ver = EKIP_VERI_SETI
    ds = rf.workspace(ws).project(proj).version(ver).download("yolov8")
    indirilenler.append((proj, ds.location, True))
    print(f"✅ Ekip veri seti indirildi: {proj}")
else:
    print("ℹ️ Ekip veri seti henüz tanımlı değil (EKIP_VERI_SETI = None).")

## 3. Birleştirme: isim tabanlı sınıf eşleme + kopya ayıklama
Her kaynağın kendi `data.yaml` dosyasındaki sınıf **adları** okunur ve hedef sınıfa çevrilir:
`head / no helmet / no-helmet / without helmet → 0 (head)`, `helmet / hardhat / hi-viz helmet → 1 (helmet)`. `person`, `vest` gibi diğer sınıflar atılır.

In [ ]:
import os, glob, shutil, hashlib, yaml
from collections import Counter, defaultdict

HEDEF_SINIFLAR = ['head', 'helmet']

def hedef_sinif(ad):
    a = ad.lower().strip().replace('_', ' ').replace('-', ' ')
    if a.startswith('no ') or 'without' in a or a in ('head', 'bare head', 'no hardhat'):
        return 0
    if 'helmet' in a or 'hardhat' in a or 'hard hat' in a:
        return 1
    if a == 'head' or a.endswith(' head'):
        return 0
    return None   # person, vest vb.

def md5(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

if os.path.exists(MERGED_DIR):
    shutil.rmtree(MERGED_DIR)
for split in ['train', 'valid', 'test']:
    os.makedirs(f"{MERGED_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{MERGED_DIR}/{split}/labels", exist_ok=True)

# Önce valid/test, sonra train işlenir: kopya bir görsel değerlendirme setindeyse
# train'e girmez (veri sızıntısı engellenir).
gorulen_hashler = {}
istatistik = defaultdict(Counter)
SPLIT_SIRASI = ['test', 'valid', 'train']

for kaynak, konum, ekip_mi in indirilenler:
    with open(f"{konum}/data.yaml") as f:
        names = yaml.safe_load(f)['names']
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    eslem = {i: hedef_sinif(n) for i, n in enumerate(names)}
    print(f"{kaynak}: " + ", ".join(f"{n}→{HEDEF_SINIFLAR[eslem[i]] if eslem[i] is not None else 'atıldı'}" for i, n in enumerate(names)))

    onek = ("EKIP_" if ekip_mi else "") + kaynak.replace('-', '_')[:20]
    for split in SPLIT_SIRASI:
        for img in glob.glob(f"{konum}/{split}/images/*"):
            h = md5(img)
            if h in gorulen_hashler:
                istatistik[split]['kopya_atlandi'] += 1
                continue
            gorulen_hashler[h] = split

            base = os.path.splitext(os.path.basename(img))[0]
            lbl = f"{konum}/{split}/labels/{base}.txt"
            yeni_satirlar = []
            if os.path.exists(lbl):
                for satir in open(lbl).read().strip().splitlines():
                    parcalar = satir.split()
                    if len(parcalar) != 5:
                        continue   # poligon/bozuk satırları atla
                    hedef = eslem.get(int(parcalar[0]))
                    if hedef is None:
                        continue
                    yeni_satirlar.append(" ".join([str(hedef)] + parcalar[1:]))
                    istatistik[split][HEDEF_SINIFLAR[hedef]] += 1

            yeni_ad = f"{onek}__{base}"
            shutil.copy(img, f"{MERGED_DIR}/{split}/images/{yeni_ad}{os.path.splitext(img)[1]}")
            with open(f"{MERGED_DIR}/{split}/labels/{yeni_ad}.txt", "w") as f:
                f.write("\n".join(yeni_satirlar))
            istatistik[split]['gorsel'] += 1
            if not yeni_satirlar:
                istatistik[split]['arka_plan(negatif)'] += 1

for split in ['train', 'valid', 'test']:
    print(f"{split:6s}: {dict(istatistik[split])}")

## 4. Sınıf dengeleme ve `data.yaml`
Hangi sınıf azınlıktaysa (baretsiz ya da baretli) o sınıfı içeren eğitim görselleri listeye bir kez daha eklenir. İki veri seti birleşince baretli örnekler ~3 kat fazla çıktığı için pratikte **baretsiz** sınıf güçlendirilir. Val/test setlerine dokunulmaz; ölçüm gerçek dağılımda yapılır.

In [ ]:
import random
random.seed(42)

# Hangi sınıf azınlıktaysa (head ya da helmet) o sınıfı içeren eğitim görselleri
# listeye tekrar eklenir. Val/test setlerine dokunulmaz.
train_imgs = sorted(glob.glob(f"{MERGED_DIR}/train/images/*"))
img_siniflari, toplam = {}, Counter()
for img in train_imgs:
    lbl = img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    siniflar = [l.split()[0] for l in open(lbl).read().splitlines() if l.strip()]
    img_siniflari[img] = siniflar
    toplam.update(siniflar)

azinlik = '0' if toplam['0'] < toplam['1'] else '1'
oran = max(toplam['0'], toplam['1']) / max(1, min(toplam['0'], toplam['1']))
tekrar = 1 if oran >= 1.5 else 0   # en fazla 1 ek kopya: eğitim süresini makul tutar
azinlik_imgs = [img for img, s in img_siniflari.items() if azinlik in s]
dengeli_liste = train_imgs + azinlik_imgs * tekrar
random.shuffle(dengeli_liste)
print(f"head örnek: {toplam['0']}, helmet örnek: {toplam['1']} (oran {oran:.2f})")
print(f"Azınlık sınıf: {HEDEF_SINIFLAR[int(azinlik)]} → {len(azinlik_imgs)} görsel +{tekrar} kez eklendi")
print(f"Eğitim listesi: {len(train_imgs)} → {len(dengeli_liste)} görsel")

with open(f"{MERGED_DIR}/train_dengeli.txt", "w") as f:
    f.write("\n".join(dengeli_liste))

data_yaml = {
    'path': MERGED_DIR,
    'train': 'train_dengeli.txt',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 2,
    'names': HEDEF_SINIFLAR,
}
with open(f"{MERGED_DIR}/data.yaml", "w") as f:
    yaml.dump(data_yaml, f, allow_unicode=True)
print(open(f"{MERGED_DIR}/data.yaml").read())

## 5. Eğitim (overfitting korumalı)
- **Model:** `yolov8s` (nano'dan ~3,5 kat büyük, daha doğru). Uygulama CPU'da 10 FPS'in altına düşerse `MODEL = 'yolov8n.pt'` yapın.
- **Süre:** 30 epoch, T4 GPU ile ~2 saat. Colab sekmesini ve bilgisayarı açık tutun.
- **Early stopping:** Doğrulama başarısı `patience` epoch boyunca artmazsa eğitim durur ve en iyi ağırlık (`best.pt`) saklanır. Ezberlemeye başlayan epoch'lar modele girmez.
- **Augmentation:** Işık, renk, ölçek ve kaydırma çeşitliliği farklı saha koşullarına dayanıklılığı artırır.

In [ ]:
from ultralytics import YOLO

MODEL = 'yolov8s.pt'
model = YOLO(MODEL)

sonuc = model.train(
    data=f"{MERGED_DIR}/data.yaml",
    epochs=30,              # ücretsiz Colab oturum süresine sığması için (~2 saat, T4)
    patience=10,            # early stopping
    imgsz=640,
    batch=16,
    device=0,
    cache='disk',           # çözülmüş görselleri diske önbelleğe al (hızlandırır)
    cos_lr=True,
    close_mosaic=8,
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,   # ışık / renk çeşitliliği
    degrees=5.0, translate=0.1, scale=0.5,
    fliplr=0.5, mosaic=1.0, mixup=0.1,
    seed=42,
    name='baret_v2',
    exist_ok=True,
)
RUN_DIR = str(sonuc.save_dir)
print("Çıktılar:", RUN_DIR)

## 6. Overfitting kontrolü
Train loss düşerken **val loss yükseliyorsa** model ezberliyordur. İki eğri birlikte düşüp yataylaşıyorsa sorun yoktur.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f"{RUN_DIR}/results.csv")
df.columns = [c.strip() for c in df.columns]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, kayip in zip(axes, ['box_loss', 'cls_loss', 'dfl_loss']):
    ax.plot(df['epoch'], df[f'train/{kayip}'], label='train')
    ax.plot(df['epoch'], df[f'val/{kayip}'], label='val')
    ax.set_title(kayip); ax.set_xlabel('epoch'); ax.legend()
plt.tight_layout(); plt.show()

en_iyi = df['metrics/mAP50-95(B)'].idxmax()
son_val = df['val/cls_loss'].iloc[-5:].mean()
min_val = df['val/cls_loss'].min()
print(f"En iyi epoch: {int(df['epoch'][en_iyi])}  |  mAP50: {df['metrics/mAP50(B)'][en_iyi]:.3f}")
if son_val > min_val * 1.15:
    print("⚠️ Son epoch'larda val cls_loss en düşük değerinden %15+ yukarıda: overfitting işareti. best.pt bu nedenle en iyi epoch'tan alınır.")
else:
    print("✅ Belirgin overfitting işareti yok.")

## 7. Test seti değerlendirmesi (modelin hiç görmediği veriler)
Raporda kullanılacak metrikler buradan alınmalıdır. Doğrulama (val) seti eğitim sırasında en iyi epoch'u seçmek için kullanıldığından tarafsız değildir.

In [ ]:
best = YOLO(f"{RUN_DIR}/weights/best.pt")
test_metrik = best.val(data=f"{MERGED_DIR}/data.yaml", split='test', conf=0.001, plots=True, name='baret_v2_test', exist_ok=True)

print(f"{'Sınıf':10s} {'Precision':>10s} {'Recall':>8s} {'mAP50':>8s} {'mAP50-95':>9s}")
for i, sinif_idx in enumerate(test_metrik.box.ap_class_index):
    p, r, ap50, ap = test_metrik.box.class_result(i)
    print(f"{HEDEF_SINIFLAR[sinif_idx]:10s} {p:10.3f} {r:8.3f} {ap50:8.3f} {ap:9.3f}")
print(f"{'Genel':10s} {test_metrik.box.mp:10.3f} {test_metrik.box.mr:8.3f} {test_metrik.box.map50:8.3f} {test_metrik.box.map:9.3f}")
print("Confusion matrix:", test_metrik.save_dir)

## 8. Önerilen güven eşiği (yanlış alarmı en aza indirmek için)
Hedef: **baretsiz (`head`) precision ≥ %95**, yani her 100 alarmın en az 95'i gerçek ihlal. Bu şartı sağlayan en düşük eşik seçilir. Bu sayede recall (yakalanan baretsiz oranı) mümkün olduğunca yüksek kalır.
Bulunan değer, uygulamadaki **güven eşiği kaydırıcısına** girilir.

In [ ]:
import numpy as np

HEDEF_PRECISION = 0.95
px = np.array(test_metrik.box.px)                 # 0..1 arası 1000 eşik noktası
p_curve = np.array(test_metrik.box.p_curve)       # [sınıf, eşik]
r_curve = np.array(test_metrik.box.r_curve)
siniflar = list(test_metrik.box.ap_class_index)
hi = siniflar.index(0)   # head
he = siniflar.index(1)   # helmet

uygun = np.where(p_curve[hi] >= HEDEF_PRECISION)[0]
if len(uygun):
    idx = uygun[0]
    print(f"✅ Önerilen eşik: {px[idx]:.2f}  →  kaydırıcıyı %{int(round(px[idx]*100))} yapın")
else:
    f1 = 2 * p_curve[hi] * r_curve[hi] / np.maximum(p_curve[hi] + r_curve[hi], 1e-9)
    idx = int(f1.argmax())
    print(f"⚠️ %{int(HEDEF_PRECISION*100)} precision hiçbir eşikte sağlanamadı. En iyi F1 eşiği: {px[idx]:.2f}. Daha fazla/çeşitli veri gerekli.")

print(f"   Bu eşikte baretsiz: precision {p_curve[hi][idx]:.3f}, recall {r_curve[hi][idx]:.3f}")
print(f"   Bu eşikte baretli : precision {p_curve[he][idx]:.3f}, recall {r_curve[he][idx]:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(px, p_curve[hi], label='head precision'); plt.plot(px, r_curve[hi], label='head recall')
plt.axvline(px[idx], color='gray', ls='--', label=f'eşik {px[idx]:.2f}')
plt.xlabel('güven eşiği'); plt.legend(); plt.title('Baretsiz (head) — eşik seçimi'); plt.show()

## 9. Modeli indir
`best.pt` dosyasını proje klasöründeki eski `best.pt` ile değiştirin.

In [ ]:
from google.colab import files
shutil.copy(f"{RUN_DIR}/weights/best.pt", "/content/best.pt")
files.download("/content/best.pt")